# Notebook 5: Evaluation and Prediction

This notebook handles model evaluation using BLEU and chrF++ metrics, and generates the final submission.

## 5.1 Setup and Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import json
from datetime import datetime

WORK_DIR = "/home/kwierman/Desktop/Projects/DeepPastAkkadian"
os.chdir(WORK_DIR)

# Check GPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Load test data
test_df = pd.read_csv("data/processed/test.csv")
val_df = pd.read_csv("data/processed/val.csv")

print(f"Test samples: {len(test_df)}")
print(f"Validation samples: {len(val_df)}")

## 5.2 Evaluation Metrics

The competition uses the geometric mean of BLEU and chrF++ scores.
Reference: [Geometric Mean of BLEU and chrF++](https://www.kaggle.com/code Competition)

In [ ]:
import sacrebleu
from sacrebleu import metrics

# Test metrics
print("SacreBLEU version:", sacrebleu.__version__)

def compute_evaluation_metrics(predictions, references):
    """
    Compute BLEU, chrF++, and their geometric mean.
    
    Note: For corpus-level evaluation, we aggregate sufficient statistics
    """
    # BLEU
    bleu = sacrebleu.corpus_bleu(predictions, [references])
    
    # chrF++
    chrf = sacrebleu.corpus_chrf(predictions, [references], order=6)
    
    # Geometric mean
    geo_mean = np.sqrt(bleu.score * chrf.score)
    
    return {
        "bleu": bleu.score,
        "chrf": chrf.score,
        "geometric_mean": geo_mean
    }

# Test on dummy data
test_preds = ["Hello world"]
test_refs = ["Hello world"]

metrics_result = compute_evaluation_metrics(test_preds, test_refs)
print(f"Test metrics: {metrics_result}")

## 5.3 Load Model for Translation

In [ ]:
from transformers import MarianTokenizer, MarianMTModel

# Check for trained model or use pre-trained
model_paths = [
    "models/akkadian-translator-final",
    "Helsinki-NLP/opus-mt-ROMANCE-en"
]

model_path = None
for path in model_paths:
    if os.path.exists(path) or path == "Helsinki-NLP/opus-mt-ROMANCE-en":
        model_path = path
        break

print(f"Using model: {model_path}")

# Load model
tokenizer = MarianTokenizer.from_pretrained(model_path)
model = MarianMTModel.from_pretrained(model_path)

if DEVICE == "cuda":
    model = model.cuda()

print(f"Model loaded: {model.num_parameters():,} parameters")

## 5.4 Translation Function

In [ ]:
def translate_batch(texts, model, tokenizer, device=DEVICE, batch_size=8, max_length=512):
    """Translate texts in batches"""
    translations = []
    
    model.eval()
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=256)
        
        if device == "cuda":
            inputs = {k: v.cuda() for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=max_length, num_beams=4)
        
        batch_translations = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translations.extend(batch_translations)
    
    return translations

print("Translation function defined.")

## 5.5 Evaluate on Validation Set

In [ ]:
# Evaluate on validation set
print("=" * 60)
print("EVALUATION ON VALIDATION SET")
print("=" * 60)

# Translate validation sources
val_sources = val_df['source'].tolist()
val_targets = val_df['target'].tolist()

print(f"Translating {len(val_sources)} validation samples...")
val_predictions = translate_batch(val_sources, model, tokenizer, batch_size=8)

# Compute metrics
val_metrics = compute_evaluation_metrics(val_predictions, val_targets)

print("\nValidation Metrics:")
print(f"  BLEU: {val_metrics['bleu']:.2f}")
print(f"  chrF++: {val_metrics['chrf']:.2f}")
print(f"  Geometric Mean: {val_metrics['geometric_mean']:.2f}")

In [ ]:
# Show some examples
print("\n" + "=" * 60)
print("SAMPLE PREDICTIONS")
print("=" * 60)

for i in range(min(3, len(val_predictions))):
    print(f"\n--- Sample {i+1} ---")
    print(f"Source: {val_sources[i][:100]}...")
    print(f"Reference: {val_targets[i][:100]}...")
    print(f"Prediction: {val_predictions[i][:100]}...")

## 5.6 Generate Test Predictions

In [ ]:
# Generate predictions for test set
print("=" * 60)
print("GENERATING TEST PREDICTIONS")
print("=" * 60)

test_sources = test_df['source'].tolist()
test_ids = test_df['id'].tolist()

print(f"Translating {len(test_sources)} test samples...")
test_predictions = translate_batch(test_sources, model, tokenizer, batch_size=8)

print(f"Generated {len(test_predictions)} predictions")

## 5.7 Create Submission File

In [ ]:
# Create submission dataframe
submission = pd.DataFrame({
    'id': test_ids,
    'translation': test_predictions
})

# Ensure proper format
print("Submission DataFrame:")
print(submission.head())

# Check format matches sample submission
sample_sub = pd.read_csv("sample_submission.csv")
print(f"\nSample submission columns: {sample_sub.columns.tolist()}")
print(f"Our submission columns: {submission.columns.tolist()}")

In [ ]:
# Save submission
output_path = "submission.csv"
submission.to_csv(output_path, index=False)

print(f"\nSubmission saved to: {output_path}")
print(f"Shape: {submission.shape}")

# Verify
verification = pd.read_csv(output_path)
print(f"Verification - shape: {verification.shape}")
print(verification.head())

## 5.8 Summary and Save Results

In [ ]:
# Save evaluation results
results = {
    "timestamp": datetime.now().isoformat(),
    "model": model_path,
    "validation_metrics": val_metrics,
    "test_samples": len(test_predictions),
    "submission_file": output_path
}

with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

summary = f"""
### Evaluation Results

**Validation Set Performance:**
- BLEU Score: {val_metrics['bleu']:.2f}
- chrF++ Score: {val_metrics['chrf']:.2f}
- Geometric Mean: {val_metrics['geometric_mean']:.2f}

**Submission:**
- File: {output_path}
- Samples: {len(test_predictions)}

### Notes
- Model: {model_path}
- Using pre-trained baseline (not fine-tuned)
- For better results, fine-tune the model following Notebook 04
"""

print(summary)

print("\nResults saved to results.json")